# Chapter 8 — Report and resolve

Engineer course · source candidate · CONVERGING

# Chapter 8 — Report and resolve

This Chapter has two self-contained halves with one shared workspace.
Execute Lesson 1 in a fresh kernel, stop it, then start a second fresh
kernel and run only Lesson 2. The web Lessons remain reading views, and
`chapter.ipynb` is a zero-output transport artifact rather than restart
evidence.

## Lesson 1 — Persist exact Results and build a report

### Build the complete request in the first kernel

This lesson owns one fresh-kernel execution. It rebuilds a coupled
grounded LC, then persists a Direct Result and a loaded diagonal-root
Result in one shared workspace.

In [ ]:
from IPython.display import display

from scnsim import (
    CircuitPlan,
    CircuitRun,
    DiagonalRootSpec,
    DirectSolveSpec,
    ParameterDefinitions,
    ParameterSet,
    ParameterSpec,
    ReductionPipeline,
    ReportSpec,
    Theme,
    components,
    units as u,
)

inputs = ParameterDefinitions(id="report_resolve_design")
capacitance = inputs.parameter(
    id="capacitance", baseline=110.0 * u.fF, spec=ParameterSpec(unit=u.fF)
)
inductance = inputs.parameter(
    id="inductance", baseline=5.8 * u.nH, spec=ParameterSpec(unit=u.nH)
)
plan = CircuitPlan(id="report_resolve_resonator")
resonator = plan.subsystem(id="resonator")
capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=capacitance)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=inductance)
)
resonator_bus = resonator.bus(id="terminal")
resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)
resonator_pin = resonator.expose_pin(id="terminal", at=resonator_bus)
resonator_coordinate = resonator.expose_coordinate(
    id="terminal_node", at=resonator_bus
)
signal_bus = plan.bus(id="signal_boundary")
coupler = plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
plan.series(
    id="coupling", start=signal_bus, elements=(coupler,), end=resonator_pin
)
plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

The child Pin is the parent wiring boundary. Its Coordinate selects
analysis on the same signal bus and creates no second electrical
connection.

In [ ]:
workspace = "workspaces/engineer-chapter-08"
run = CircuitRun(plan=plan, workspace=workspace)
direct_spec = DirectSolveSpec(frequencies=[5.5, 6.0, 6.5, 7.0] * u.GHz)
quantity_view = run.original.reduce(
    ReductionPipeline().retain(resonator_coordinate)
)
root_spec = DiagonalRootSpec(
    coordinate=resonator_coordinate,
    root_hint=6.0 * u.GHz,
)
baseline_parameters = ParameterSet()
run.explain(quantity_view, root_spec, parameters=baseline_parameters).show()

In [ ]:
direct = run.solve(run.original, direct_spec, parameters=baseline_parameters)
root = run.evaluate(quantity_view, root_spec, parameters=baseline_parameters)
display(direct.s.show(magnitude="db", theme=Theme.AUTO))
display(root.show())

The Direct request uses exactly four declared samples. The loaded root
uses a 6 GHz Newton hint; the hint is not the answer or a search
interval.

In [ ]:
report = run.build_report(
    ReportSpec(inputs=(direct, root), theme=Theme.AUTO)
)
report.show()

`ReportSpec` receives the two explicit immutable Results. Building
presentation does not choose another request or create a numerical
attempt.

## Lesson 2 — Restart, reconstruct, and resolve only

### Start a second independent kernel

Stop the first kernel. Start a new Python kernel in the same preserved
working directory, then run only the three cells in this Lesson. They
deliberately repeat the complete declaration and request. Do not run
Lesson 1’s execution cells in the second kernel.

In [ ]:
from IPython.display import display

from scnsim import (
    CircuitPlan,
    CircuitRun,
    DiagonalRootSpec,
    ParameterDefinitions,
    ParameterSet,
    ParameterSpec,
    ReductionPipeline,
    components,
    units as u,
)

restart_inputs = ParameterDefinitions(id="report_resolve_design")
restart_capacitance = restart_inputs.parameter(
    id="capacitance", baseline=110.0 * u.fF, spec=ParameterSpec(unit=u.fF)
)
restart_inductance = restart_inputs.parameter(
    id="inductance", baseline=5.8 * u.nH, spec=ParameterSpec(unit=u.nH)
)
restart_plan = CircuitPlan(id="report_resolve_resonator")
restart_resonator = restart_plan.subsystem(id="resonator")
restart_capacitor = restart_resonator.add(
    components.capacitor(id="capacitor", capacitance=restart_capacitance)
)
restart_inductor = restart_resonator.add(
    components.inductor(id="inductor", inductance=restart_inductance)
)
restart_resonator_bus = restart_resonator.bus(id="terminal")
restart_resonator.parallel(
    id="parallel_lc",
    start=restart_resonator_bus,
    branches=((restart_capacitor,), (restart_inductor,)),
    end=restart_resonator.ground,
)
restart_pin = restart_resonator.expose_pin(
    id="terminal", at=restart_resonator_bus
)
restart_coordinate = restart_resonator.expose_coordinate(
    id="terminal_node", at=restart_resonator_bus
)
restart_signal_bus = restart_plan.bus(id="signal_boundary")
restart_coupler = restart_plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
restart_plan.series(
    id="coupling",
    start=restart_signal_bus,
    elements=(restart_coupler,),
    end=restart_pin,
)
restart_plan.add_port(
    id="signal_in",
    at=restart_signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

In [ ]:
restart_workspace = "workspaces/engineer-chapter-08"
restart_run = CircuitRun(plan=restart_plan, workspace=restart_workspace)
restart_view = restart_run.original.reduce(
    ReductionPipeline().retain(restart_coordinate)
)
restart_root_spec = DiagonalRootSpec(
    coordinate=restart_coordinate,
    root_hint=6.0 * u.GHz,
)
restart_parameters = ParameterSet()

### Resolve without execution fallback

In [ ]:
resolved_root = restart_run.resolve(
    restart_view,
    restart_root_spec,
    parameters=restart_parameters,
)
resolved_root.show()

The second kernel contains no `solve()`, `evaluate()`, retry, or
fallback. It reconstructs the exact request identity and loads verified
evidence from the shared workspace. Running every cell sequentially in
one ordinary kernel does not prove this restart contract; the
source-bound generation receipt records the two independent kernel
sessions.